In [0]:
%run ./utility/logger

In [0]:
dbutils.widgets.text('catalog',"")
dbutils.widgets.text('schema',"")
dbutils.widgets.text('env',"")

In [0]:
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
env = dbutils.widgets.get('env')

In [0]:
print(schema)

In [0]:
spark.sql(f"""CREATE OR REPLACE TEMP VIEW customer_incremental AS
SELECT *
FROM commerce_stage_{env}.silver.customer_stage
WHERE updated_ts >
(
SELECT COALESCE(MAX(last_processed_ts), TIMESTAMP('1900-01-01'))
FROM {catalog}.util.etl_control
WHERE table_name = 'dim_customer'
)
""")

In [0]:
%sql
select * from customer_incremental

In [0]:
spark.sql(f"""
MERGE INTO {catalog}.{schema}.dim_customer tgt
USING customer_incremental src

ON tgt.customer_id = src.customer_id
AND tgt.is_current = true

WHEN MATCHED
AND (
COALESCE(tgt.customer_name,'') <> COALESCE(src.customer_name,'')
OR COALESCE(tgt.email,'') <> COALESCE(src.email,'')
OR COALESCE(tgt.city,'') <> COALESCE(src.city,'')
OR COALESCE(tgt.state,'') <> COALESCE(src.state,'')
)

THEN UPDATE SET
tgt.is_current = false,
tgt.effective_to = current_timestamp(),
tgt.updated_ts = current_timestamp()
""");

In [0]:
spark.sql(f"""
INSERT INTO {catalog}.{schema}.dim_customer
(
customer_id,
customer_name,
email,
city,
state,
effective_from,
effective_to,
is_current,
created_ts,
updated_ts
) 
SELECT
src.customer_id,
src.customer_name,
src.email,
src.city,
src.state,
current_timestamp(),
TIMESTAMP('9999-12-31 23:59:59'),
true,
current_timestamp(),
current_timestamp()

FROM customer_incremental src

LEFT JOIN {catalog}.{schema}.dim_customer tgt
ON src.customer_id = tgt.customer_id
AND tgt.is_current = true
WHERE tgt.customer_id IS NULL
""");

In [0]:
spark.sql(f"""
MERGE INTO {catalog}.util.etl_control tgt
USING
(
SELECT
'dim_customer' AS table_name,
MAX(updated_ts) AS last_processed_ts
FROM commerce_stage_{env}.silver.customer_stage
) src

ON tgt.table_name = src.table_name

WHEN MATCHED THEN
UPDATE SET
tgt.last_processed_ts = src.last_processed_ts

WHEN NOT MATCHED THEN
INSERT
(
table_name,
last_processed_ts
)
VALUES
(
src.table_name,
src.last_processed_ts
)
""");